# 📊🤖 Mesa Financiera en Telegram — Ruta A (Google Colab)
**Dra. Elda C. Morales · Actividad para principiantes**

Construirás un ecosistema de **3 agentes** que trabajan en cadena, dirigidos por **Batuta**, el orquestador:

| Agente | Recibe | Hace | Entrega |
|---|---|---|---|
| 🛰️ **Iris · La Recolectora** (Agente 1) | Ticker que escribes en Telegram | Consulta datos reales de Alpha Vantage mediante **MCP** | Ficha JSON |
| 🧮 **Euler · Analista Financiero** (Agente 2) | La ficha de Iris | Python **calcula**; OpenAI interpreta con prudencia metodológica | Análisis JSON |
| ✍️ **Vera · Editora Financiera** (Agente 3) | El análisis de Euler | Jerarquiza el hallazgo y redacta una nota clara | Mensaje en Telegram |

**Cómo usar este notebook:** ejecuta las celdas en orden con ▶ (o `Shift + Enter`). Después de cada celda revisa el apartado *"Deberías ver"*.

> 💡 ¿Aún no tienes llaves? Deja `MODO_SIMULACION = True` en la celda 2: todo funciona con datos ficticios.

## Celda 1 · Instalar librerías
**Deberías ver:** la celda termina sin errores en rojo (puede tardar unos 20 segundos).

In [ ]:
!pip install -q openai requests

## Celda 2 · Configuración y llaves
Las llaves **nunca** se escriben en el código. Guárdalas en el panel **🔑 Secretos** (barra izquierda de Colab) → *Add new secret*, con estos nombres exactos y el interruptor **Notebook access** encendido:

`TELEGRAM_BOT_TOKEN` · `TELEGRAM_CHAT_ID` · `OPENAI_API_KEY` · `ALPHAVANTAGE_API_KEY`

**Deberías ver:** una tabla con ✅ en cada llave configurada y el modo (SIMULACIÓN o REAL).

In [ ]:
import json, os, re, time
import requests

MODO_SIMULACION = True        # ← cámbialo a False cuando tengas tus 4 llaves
MODO_CLASE      = True        # True = el bot muestra cada entrega entre agentes
MODELO          = "gpt-5.5"

def leer_secreto(nombre):
    """Lee un secreto de Colab (o de una variable de entorno fuera de Colab)."""
    try:
        from google.colab import userdata
        return (userdata.get(nombre) or "").strip()
    except Exception:
        return os.environ.get(nombre, "").strip()

TELEGRAM_TOKEN   = leer_secreto("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = leer_secreto("TELEGRAM_CHAT_ID")
OPENAI_KEY       = leer_secreto("OPENAI_API_KEY")
ALPHAVANTAGE_KEY = leer_secreto("ALPHAVANTAGE_API_KEY")

if not MODO_SIMULACION and not (OPENAI_KEY and ALPHAVANTAGE_KEY):
    print("⚠️ Faltan OPENAI_API_KEY o ALPHAVANTAGE_API_KEY: activo el MODO SIMULACIÓN.")
    MODO_SIMULACION = True

for nombre, valor in [("TELEGRAM_BOT_TOKEN", TELEGRAM_TOKEN), ("TELEGRAM_CHAT_ID", TELEGRAM_CHAT_ID),
                      ("OPENAI_API_KEY", OPENAI_KEY), ("ALPHAVANTAGE_API_KEY", ALPHAVANTAGE_KEY)]:
    print(f"{nombre:<22} {'✅ configurada' if valor else '⬜ vacía'}")   # nunca se imprime el valor
print("Modo:", "SIMULACIÓN (datos ficticios)" if MODO_SIMULACION else "REAL")

## Celda 3 · Conexión con OpenAI y con el servidor MCP
Aquí ocurre la integración **MCP**: la Responses API de OpenAI usa el servidor remoto de Alpha Vantage para consultar datos bursátiles.

**Deberías ver:** `✅ Conexión preparada`.

In [ ]:
DESCARGO = "No constituye una recomendación de inversión."
cliente = None
if not MODO_SIMULACION:
    from openai import OpenAI
    cliente = OpenAI(api_key=OPENAI_KEY)

def llamar_openai(system, mensaje, usar_mcp=False, max_tokens=900):
    """Envía un mensaje con Responses API. Devuelve texto y herramientas MCP usadas."""
    base = dict(model=MODELO, max_output_tokens=max_tokens, instructions=system,
                input=mensaje, store=False)
    if usar_mcp:
        resp = cliente.responses.create(
            **base,
            tools=[{"type": "mcp", "server_label": "alphavantage",
                    "server_url": f"https://mcp.alphavantage.co/mcp?apikey={ALPHAVANTAGE_KEY}",
                    "require_approval": "never"}])
    else:
        resp = cliente.responses.create(**base)
    texto = resp.output_text.strip()
    return texto, [getattr(item, "name", "alphavantage") for item in resp.output
                   if getattr(item, "type", "") == "mcp_call"]

def extraer_json(texto):
    inicio, fin = texto.find("{"), texto.rfind("}")
    if inicio == -1 or fin <= inicio:
        raise ValueError("El agente no devolvió JSON")
    return json.loads(texto[inicio:fin + 1])

def a_numero(valor):
    try:
        return float(str(valor).replace("%", "").replace(",", ""))
    except (TypeError, ValueError):
        return None

print("✅ Conexión preparada")

## Celda 4 · 🛰️ Iris, la Recolectora (Agente 1)
Curiosa y meticulosa: pide a OpenAI que consulte Alpha Vantage por MCP y devuelva una **ficha JSON**. Si el MCP falla o se agota la cuota diaria, usa datos DEMO para que la clase no se detenga.

**Deberías ver:** `✅ Iris lista`.

In [ ]:
PROMPT_RECOLECTOR = """Eres Iris, la Recolectora (Agente 1) de una mesa financiera educativa.
Tu personalidad: curiosa y meticulosa; jamás inventas un dato.
Tarea: obtener datos REALES de la acción indicada con las herramientas de Alpha Vantage.
Reglas:
- Haz como máximo 2 consultas: cotización actual y datos generales de la empresa.
- NUNCA inventes cifras. Si un dato no está disponible, escribe null.
- Conserva la fecha informada por la fuente.
- No combines datos reales con datos simulados.
- No calcules, interpretes ni emitas recomendaciones.
- Responde SOLO con un JSON con estas claves exactas:
  ticker, nombre, sector, precio, cambio_pct, fecha_dato, pe_ratio, beta,
  max_52s, min_52s, cap_mercado_usd"""

DATOS_DEMO = {"nombre": "Empresa ficticia (simulación)", "sector": "Tecnología", "precio": 150.0,
              "cambio_pct": 1.2, "fecha_dato": "dato ficticio", "pe_ratio": 28.0, "beta": 1.25,
              "max_52s": 180.0, "min_52s": 120.0, "cap_mercado_usd": 2400000000000}

def agente_recolector(ticker):
    if MODO_SIMULACION:
        return {"ticker": ticker, **DATOS_DEMO, "fuente": "DATOS DEMO, no son reales (modo simulación)"}, []
    try:
        texto, herramientas = llamar_openai(PROMPT_RECOLECTOR, f"Acción: {ticker}", usar_mcp=True)
        ficha = extraer_json(texto)
        ficha.update(ticker=ticker, fuente="Alpha Vantage vía MCP")
        return ficha, herramientas
    except Exception as e:
        print("⚠️ MCP no disponible, uso datos DEMO:", type(e).__name__)
        return {"ticker": ticker, **DATOS_DEMO, "fuente": "DATOS DEMO, no son reales (MCP no disponible)"}, []

print("✅ Iris lista")

## Celda 5 · 🧮 Euler, el Analista Financiero (Agente 2)
**Regla de oro:** los números los calcula el código; la IA solo interpreta.

- Posición en el rango de 52 semanas: $P_{52} = \dfrac{P_t - P_{\min}}{P_{\max} - P_{\min}} \times 100$
- Semáforo de beta (umbrales didácticos): 🟢 β < 0.8 · 🟡 0.8 ≤ β ≤ 1.2 · 🔴 β > 1.2

**Deberías ver:** `{'posicion_rango_52s_pct': 50.5, ...}` y `✅ Euler listo`.

In [ ]:
def calcular_indicadores(ficha):
    precio, maxi, mini = (a_numero(ficha.get(k)) for k in ("precio", "max_52s", "min_52s"))
    beta, pe = a_numero(ficha.get("beta")), a_numero(ficha.get("pe_ratio"))
    posicion = None
    if None not in (precio, maxi, mini) and maxi > mini:
        posicion = round((precio - mini) / (maxi - mini) * 100, 1)
    distancia = round((precio / maxi - 1) * 100, 1) if precio is not None and maxi not in (None, 0) else None
    avance = round((precio / mini - 1) * 100, 1) if precio is not None and mini not in (None, 0) else None
    recuperacion = round((maxi / precio - 1) * 100, 1) if precio not in (None, 0) and maxi is not None else None
    if posicion is None: zona = "sin posición disponible"
    elif posicion < 0 or posicion > 100: zona = "fuera del rango informado; verificar fechas"
    elif posicion <= 20: zona = "cerca del mínimo"
    elif posicion <= 40: zona = "zona baja"
    elif posicion <= 60: zona = "zona media"
    elif posicion <= 80: zona = "zona alta"
    else: zona = "cerca del máximo"
    if beta is None: semaforo = "⚪ sin dato de beta"
    elif beta < 0.8: semaforo = "🟢 volatilidad baja"
    elif beta <= 1.2: semaforo = "🟡 volatilidad similar al mercado"
    else: semaforo = "🔴 volatilidad alta"
    return {"posicion_rango_52s_pct": posicion, "zona_rango_52s": zona,
            "distancia_max_52s_pct": distancia, "avance_desde_min_52s_pct": avance,
            "recuperacion_hasta_max_52s_pct": recuperacion, "semaforo_beta": semaforo,
            "beta": beta, "pe_ratio": pe, "cambio_pct": a_numero(ficha.get("cambio_pct"))}

PROMPT_ANALISTA = """Eres Euler, el Analista Financiero (Agente 2).
Tu personalidad es prudente, analítica y metodológicamente estricta.
Recibes la ficha de Iris y los indicadores calculados por Python.
No recalcules ni cambies cifras. Una sesión no confirma tendencia; beta no equivale a riesgo total;
P/E requiere comparables y capitalización describe tamaño, no seguridad.
El hallazgo debe combinar el cambio de la última sesión con el contexto de 52 semanas, ser verificable
y no repetir beta ni P/E. No uses lenguaje de compra, venta u oportunidad.
Responde SOLO con JSON: hallazgo_principal, lectura_riesgo, lectura_valuacion,
dato_pendiente, pregunta_para_investigar."""

def lectura_simulada(ind, ficha):
    cambio, distancia = ind["cambio_pct"], ind["distancia_max_52s_pct"]
    if ind["posicion_rango_52s_pct"] is None or cambio is None:
        hallazgo = "No hay datos suficientes para relacionar el movimiento reciente con el rango de 52 semanas."
    else:
        verbo = "avanzó" if cambio > 0 else "cayó" if cambio < 0 else "no cambió"
        bajo_max = f" y {abs(distancia):.1f} % por debajo del máximo" if distancia is not None and distancia <= 0 else ""
        ubicacion = "a mitad" if ind['posicion_rango_52s_pct'] == 50 else f"en la {ind['zona_rango_52s']}"
        hallazgo = (f"{ficha['ticker']} {verbo} {abs(cambio):.2f} % en la última sesión, pero se mantiene "
                     f"{ubicacion} de su rango de 52 semanas{bajo_max}. "
                     "Este movimiento aislado no permite confirmar una tendencia.")
    beta, pe = ind['beta'], ind['pe_ratio']
    if beta is None: riesgo = "No hay beta disponible para describir su sensibilidad histórica frente al mercado."
    elif beta < 0.8: riesgo = "Menor sensibilidad histórica frente al mercado; no representa por sí sola el riesgo total."
    elif beta <= 1.2: riesgo = "Sensibilidad histórica similar al mercado; no representa por sí sola el riesgo total."
    else: riesgo = "Mayor sensibilidad histórica frente al mercado; no representa por sí sola el riesgo total."
    valuacion = ("No hay P/E disponible para esta lectura." if pe is None else
                 f"El precio representa {pe:.1f} veces las utilidades por acción consideradas por la fuente. "
                 "Requiere comparación con empresas similares.")
    return {"hallazgo_principal": hallazgo,
            "lectura_riesgo": riesgo, "lectura_valuacion": valuacion,
            "dato_pendiente": "Comparar la valuación con empresas de tamaño y actividad similares.",
            "pregunta_para_investigar": "¿Cómo se compara su P/E con empresas del mismo sector, tamaño y actividad?"}

def agente_analista(ficha):
    ind = calcular_indicadores(ficha)
    if MODO_SIMULACION or str(ficha.get("fuente", "")).startswith("DATOS DEMO"):
        lectura = lectura_simulada(ind, ficha)
    else:
        texto, _ = llamar_openai(PROMPT_ANALISTA, json.dumps({"ficha": ficha, "indicadores": ind}, ensure_ascii=False))
        lectura = extraer_json(texto)
    return {"ficha": ficha, "indicadores": ind, "lectura": lectura}

print(calcular_indicadores({"precio": 250.5, "max_52s": 300, "min_52s": 200, "beta": 0.7}))
print("✅ Euler listo")

## Celda 6 · ✍️ Vera, la Editora Financiera (Agente 3)
Clara, selectiva, serena y orientada a comunicación financiera. **Deberías ver:** `✅ Vera lista`.

In [ ]:
PROMPT_COMUNICADOR = f"""Eres Vera, la Editora Financiera (Agente 3).
Tu personalidad es clara, selectiva, serena y orientada a comunicación financiera.
Jerarquiza el hallazgo sin cambiar cifras ni agregar información. No presentes una sesión como tendencia,
beta como riesgo total, P/E como caro o barato sin comparables, ni capitalización como seguridad.
No uses lenguaje de compra, venta u oportunidad. Distingue datos reales y simulados.
Entrega texto plano, máximo 1,300 caracteres, y cierra con esta frase exacta: {DESCARGO}"""

def formatear_numero(valor, decimales=1):
    numero = a_numero(valor)
    return "no disponible" if numero is None else f"{numero:.{decimales}f}"

def formatear_capitalizacion(valor):
    numero = a_numero(valor)
    return "no disponible" if numero is None else f"USD {numero / 1_000_000_000:,.0f} mil millones"

def agente_comunicador(analisis):
    f, ind, lec = analisis["ficha"], analisis["indicadores"], analisis["lectura"]
    if MODO_SIMULACION or str(f.get("fuente", "")).startswith("DATOS DEMO"):
        cambio = "no disponible" if ind["cambio_pct"] is None else f"{ind['cambio_pct']:+.2f} %"
        return (f"✍️ Vera | Mesa Financiera
⚠️ SIMULACIÓN · Datos ficticios

"
                f"📊 {f['ticker']} | Precio, riesgo y valuación
"
                f"💵 Precio: USD {formatear_numero(f.get('precio'), 2)} | Cambio última sesión: {cambio}

"
                f"💡 Hallazgo principal
{lec['hallazgo_principal']}

"
                f"📍 Contexto de 52 semanas
• Mínimo: USD {formatear_numero(f.get('min_52s'), 2)} | "
                f"Máximo: USD {formatear_numero(f.get('max_52s'), 2)}
"
                f"• Posición en el rango: {formatear_numero(ind.get('posicion_rango_52s_pct'))} %

"
                f"⚖️ Riesgo
• Beta: {formatear_numero(ind.get('beta'), 2)} | {lec['lectura_riesgo']}

"
                f"🏢 Perfil y valuación
• Sector: {f.get('sector') or 'no disponible'} | "
                f"Capitalización: {formatear_capitalizacion(f.get('cap_mercado_usd'))}
"
                f"• P/E: {formatear_numero(ind.get('pe_ratio'))}x | {lec['lectura_valuacion']}

"
                f"🔎 Para profundizar
{lec['pregunta_para_investigar']}

"
                f"🕒 Dato ficticio | Fuente: simulación local

"
                f"Datos simulados con fines educativos. {DESCARGO}")
    texto, _ = llamar_openai(PROMPT_COMUNICADOR, json.dumps(analisis, ensure_ascii=False), max_tokens=600)
    texto = texto if DESCARGO in texto else f"{texto}

{DESCARGO}"
    return f"✍️ Vera | Mesa Financiera
{texto}"

print("✅ Vera lista")

## Celda 7 · 🎼 Batuta, el orquestador, y una prueba sin Telegram
Batuta decide el orden y **pasa el output de cada agente como input del siguiente**.

**Deberías ver:** las entregas `1→2` y `2→3` y, al final, la nota con el descargo de responsabilidad.
En modo real, la ficha debe decir `"fuente": "Alpha Vantage vía MCP"`. Toma **captura** de esta salida.

In [ ]:
def pipeline(ticker, avisar=print):
    ticker = ticker.strip().upper()
    if not re.fullmatch(r"[A-Z0-9.\-]{1,12}", ticker):
        raise ValueError(f"Ticker no válido: {ticker}")
    avisar(f"🎼 Batuta · Orquestador\nRecibí tu orden. Inicio la cadena para {ticker}.")
    avisar(f"🛰️ Iris · Recolectora buscando datos de {ticker}…")
    ficha, herramientas = agente_recolector(ticker)
    if MODO_CLASE:
        avisar("📦 Entrega 1→2 · Iris → Euler (ficha):\n" + json.dumps(ficha, ensure_ascii=False, indent=1)[:1500]
               + f"\n🔧 Herramientas MCP usadas: {', '.join(herramientas) or 'ninguna'}")
    avisar("🧮 Euler · Analista Financiero calculando e interpretando…")
    analisis = agente_analista(ficha)
    if MODO_CLASE:
        avisar("📦 Entrega 2→3 · Euler → Vera (indicadores):\n" + json.dumps(analisis["indicadores"], ensure_ascii=False, indent=1))
    avisar("✍️ Vera · Editora Financiera jerarquizando y redactando…")
    return agente_comunicador(analisis)

print(pipeline("IBM"))

## Celda 8 · 📱 Telegram como centro de mando
Al ejecutar esta celda el bot queda **escuchando**. Escríbele desde Telegram: `/analiza AAPL`.

- Para apagarlo, pulsa ■ (detener) en la celda.
- Si cierras la pestaña, el bot se apaga. Para que funcione con la computadora apagada usa la **Ruta B (GitHub Actions)**.

**Deberías ver:** `✅ Bot @tu_bot escuchando` y, en Telegram, los mensajes de Batuta, Iris, Euler y Vera.

In [ ]:
API_TG = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}"
AYUDA = ("🤖 Mesa Financiera de 3 agentes\n/analiza TICKER → ejemplo: /analiza AAPL\n"
         "Batuta dirige: Iris (MCP) → Euler → Vera")

def enviar(chat_id, texto):
    requests.post(f"{API_TG}/sendMessage", data={"chat_id": chat_id, "text": texto[:4000]}, timeout=20)

def atender(mensaje):
    chat_id, texto = mensaje["chat"]["id"], (mensaje.get("text") or "").strip()
    if TELEGRAM_CHAT_ID and str(chat_id) != TELEGRAM_CHAT_ID:
        return enviar(chat_id, "Este bot es privado.")
    if not texto.lower().startswith("/analiza"):
        return enviar(chat_id, AYUDA)
    partes = texto.split()
    try:
        nota = pipeline(partes[1] if len(partes) > 1 else "", avisar=lambda t: enviar(chat_id, t))
        enviar(chat_id, nota)
    except ValueError:
        enviar(chat_id, "Escribe un ticker válido. Ejemplo: /analiza MSFT")
    except Exception as e:
        enviar(chat_id, f"⚠️ No pude completar el análisis ({type(e).__name__}). Revisa llaves y saldo.")

def encender_bot():
    yo = requests.get(f"{API_TG}/getMe", timeout=20).json()
    if not yo.get("ok"):
        raise RuntimeError("❌ Token de Telegram inválido o vacío. Revísalo con @BotFather.")
    requests.get(f"{API_TG}/deleteWebhook", timeout=20)
    print(f"✅ Bot @{yo['result']['username']} escuchando. Pulsa ■ para apagarlo.")
    offset = None
    try:
        while True:
            try:
                r = requests.get(f"{API_TG}/getUpdates", params={"timeout": 30, "offset": offset}, timeout=45).json()
                for upd in r.get("result", []):
                    offset = upd["update_id"] + 1
                    if "message" in upd:
                        atender(upd["message"])
            except requests.RequestException as e:
                print("Conexión inestable, reintento en 5 s:", e)
                time.sleep(5)
    except KeyboardInterrupt:
        print("👋 Bot detenido.")

if TELEGRAM_TOKEN:
    encender_bot()
else:
    print("⬜ Falta TELEGRAM_BOT_TOKEN: crea tu bot con @BotFather y guárdalo en 🔑 Secretos.")

## ✅ Lista de validación
- [ ] La celda 7 muestra las entregas 1→2 y 2→3.
- [ ] En modo real, la fuente dice **Alpha Vantage vía MCP** y aparecen herramientas usadas.
- [ ] En Telegram, `/analiza AAPL` devuelve tres avisos y una nota final con el descargo.
- [ ] `/analiza AAPL;x` responde "Escribe un ticker válido".

> Material educativo de la Dra. Elda C. Morales. Las notas **no son recomendaciones de inversión**.